In [2]:
# Loading the config toml file
from pathlib import Path
import tomllib

config_path = Path('config.toml')

if not config_path.exists():
    print("config.toml not found. Please  create a configuration file.")
    raise FileNotFoundError("config.toml not found")

with open(config_path, 'rb') as f:
    config = tomllib.load(f)

In [3]:
# process files
import os
from models import CrawledFile
from typing import List
from datetime import datetime

data_path = config['paths']['input']
    
crawled_files: List[CrawledFile] = []
supported_extensions = set(config['supported_extensions'])

for root, _, files in os.walk(data_path):
    for file in files:
        file_path = os.path.join(root, file)
        extension = file.split('.')[-1].lower()

        if extension not in supported_extensions:
            print(f"Warning: Unsupported file extension: {extension}. Do you want to manually add {file} to the config?")
            continue
        try:
            # Check if file is readable
            if not os.access(file_path, os.R_OK):
                print(f"Warning: No read permission for {file_path}.")
                # TODO add to excluded files list
                # Stop processing if we encounter a non-readable file
                break
            
            # Get basic file info without loading content
            file_info = {
                'filepath': file_path,
                'extension': extension,
                'size': os.path.getsize(file_path),
                'crawl_date': datetime.now().isoformat(),
            }
            crawled_files.append(CrawledFile(**file_info))
        except Exception as e:
            print(f"Error processing file {file_path}: {e}")

print(f"Discovered {len(crawled_files)} files.")

Discovered 3 files.


In [7]:
# files_to_process = discover_files(config)
files_to_process = crawled_files 
bank_files = {'HDFC': [], 'SBI': [], 'AXIS': [], 'UNKNOWN': []}

for file in files:
    filename_lower = file.filename.lower()
    if 'hdfc' in filename_lower:
        bank_files['HDFC'].append(file)
    elif 'sbi' in filename_lower:
        bank_files['SBI'].append(file)
    elif 'axis' in filename_lower:
        bank_files['AXIS'].append(file)
    else:
        bank_files['UNKNOWN'].append(file)
      
# bank_files = consolidate_files_by_bank(files_to_process)

In [8]:
for bank, files in bank_files.items():
    if files:
        print(f"\nBank: {bank}")
        for f in files:
            print(f"  - {f.filename}")

In [10]:
dataframes = {}
    
for file in files:
    try:
        if file.extension == 'csv':
            df = pd.read_csv(file.filename)
            dataframes[file.filename] = df
            print(f"Loaded CSV: {file.filename} with {len(df)} rows")
            
        elif file.extension.lower() in ['xls', 'xlsx']:
            # Load all sheets from Excel file
            excel_file = pd.ExcelFile(file.filename)
            for sheet_name in excel_file.sheet_names:
                df = pd.read_excel(file.filename, sheet_name=sheet_name)
                key = f"{file.filename}:{sheet_name}"
                dataframes[key] = df
                print(f"Loaded Excel sheet: {key} with {len(df)} rows")
                
        elif file.extension == 'pdf':
            # PDF processing is currently skipped
            print(f"PDF file skipped: {file.filename}")
            continue
            
    except Exception as e:
        print(f"Error loading file {file.filename}: {e}")

In [ ]:
# """Analyze loaded dataframes and provide insights"""
if not dataframes:
    print("No dataframes to analyze.")

print(f"\n=== Dataframe Analysis ===")
print(f"Total dataframes: {len(dataframes)}")

for key, df in dataframes.items():
    print(f"\nDataframe: {key}")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Data types:")
    for col, dtype in df.dtypes.items():
        print(f"    {col}: {dtype}")
    
    # Show first few rows
    if len(df) > 0:
        print(f"  First 3 rows:")
        print(df.head(3).to_string())
    else:
        print(f"  Empty dataframe")

In [ ]:
try:
    # Analyze loaded dataframes
    if dataframes:        
        # Example: Get dataframes for a specific bank
        hdfc_dataframes = get_dataframe_by_bank(dataframes, 'HDFC')
        if hdfc_dataframes:
            print(f"\nHDFC dataframes: {len(hdfc_dataframes)}")
            for key in hdfc_dataframes.keys():
                print(f"  - {key}")
except FileNotFoundError:
    print("Configuration file not found. Please ensure config.toml exists.")
    exit(1)
except Exception as e:
    print(f"An error occurred: {e}")
    exit(1)